<!-- dads-lab-header -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M12/M12_Lab1_OpenAI_Agents_SDK.ipynb)

![Module 12 Lab 1 - OpenAI Agents SDK](https://raw.githubusercontent.com/mdehghani86/DADS5250-GenAI/main/labs/M12/assets/images/M12_Lab1_OpenAI_Agents_SDK_banner.png)

In [ ]:
# === Shared lab setup: install dads5250 + the OpenAI Agents SDK, then import ===
# Installs the shared utilities (pp, pretty_print, lab_pill, model constants,
# setup_openai) AND the `openai-agents` package once per Colab runtime. The
# same OPENAI_API_KEY Colab secret is used across every DADS 5250 lab — set it
# once in the 🔑 sidebar and it is picked up automatically.
import os
import importlib.util
# One combined install: the shared utils package + the Agents SDK it drives
!pip install -q "git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils" openai-agents

from dads5250 import (
    pp,
    pretty_print,
    lab_pill,
    setup_openai,
    DEFAULT_CHAT_MODEL,   # newest reasoning model that supports temperature
    DEFAULT_MINI_MODEL,   # newest mini model — fast + cheap, what our agents use
)

lab_pill('M12 Lab 1 — OpenAI Agents SDK')   # sticky banner so you always see which lab you're in

## API check

Confirm the API connection before we start. `setup_openai()` loads your key and, importantly, also writes it into `os.environ["OPENAI_API_KEY"]`. The Agents SDK reads that environment variable directly, so once this cell runs green, every `Agent` and `Runner` below can talk to the model with no extra configuration. Your key comes from a Colab Secret, an environment variable, or a hidden prompt if neither is set.

In [ ]:
# === API check: confirm the connection and expose the key to the Agents SDK ===
client = setup_openai()                       # loads + verifies OPENAI_API_KEY
os.environ["OPENAI_API_KEY"] = client.api_key # the Agents SDK reads the key from the environment

pp({
    "OpenAI":       "connected",
    "agents SDK":   "reads OPENAI_API_KEY from os.environ",
    "agent model":  DEFAULT_MINI_MODEL,        # the model our agents will run on
}, title="API check")

# 🤖 From Tools to Teams: the OpenAI Agents SDK

**Why this matters.** In Module 3 you wired a model to a single tool by hand: you wrote the JSON schema, sent the `tools` list, caught the tool call, ran the function, and fed the result back in a loop. That taught you *exactly* what is happening under the hood, and it works. But writing that call-execute-return loop yourself for every project, then bolting on retries, routing between agents, and safety checks, quickly becomes a lot of plumbing.

**The Agents SDK is that plumbing, done for you.** It is a small, opinionated Python framework (the production successor to OpenAI's experimental "Swarm") that turns the loop you hand-wrote into a handful of primitives. You describe *what* your system is — an **Agent** with instructions, some **Tools**, maybe some **Handoffs** to specialists, and **Guardrails** for safety — and a **Runner** executes the whole agentic loop for you and hands back the final answer.

**Minimalism is the point.** The SDK deliberately has a tiny surface area: a few powerful pieces you compose with plain Python. If you know Python functions and classes, you already know most of it. In this lab we climb the ladder one rung at a time: your first agent, then tools, then a multi-agent handoff, then a brief look at guardrails. By the end you can stand up a small multi-agent app in a few lines.

## 🧩 1. What the Agents SDK is (and why, vs the raw API)

Strip the SDK down and it is four nouns and one verb:

- **Agent** — a model configured with a `name`, `instructions` (its job description), and optionally `tools` and `handoffs`.
- **Tools** — plain Python functions the agent may call, marked with a `@function_tool` decorator so the SDK can read their name, arguments, and docstring automatically.
- **Handoffs** — other agents this agent is allowed to delegate to, so one coordinator can route work to specialists.
- **Guardrails** — input/output checks that run alongside the agent and fail fast on unsafe or off-topic requests.
- **Runner** (the verb) — the execution engine. It takes an agent plus an input, runs the tool-calling and handoff loop until the task is done, and returns a result.

**Why not just use the raw Chat Completions API (like M3)?** You *can*, and you did. The difference is who writes the loop. With the raw API, **you** manage the message list, detect tool calls, dispatch functions, append results, and re-call the model — for every tool and every turn. The SDK folds all of that into `Runner.run_sync(...)`. It also builds the tool JSON schema *from your function's type hints and docstring*, so you never hand-write a schema again. Same mechanics you already understand — far less boilerplate.

## 🚀 2. Your first agent (Agent + Runner)

The smallest useful program is an **Agent** and a **Runner**. The Agent is a description; the Runner makes it go.

You give the agent a `name` (for logs and handoffs), `instructions` (its system prompt, in plain English), and a `model`. Then `Runner.run_sync(agent, "...")` runs the agentic loop synchronously and returns a `result` object. The answer lives in `result.final_output`. No tools yet — just prove the wiring works end to end.

In [ ]:
# ==========================================================
# 2. Your first agent: Agent + Runner
# ==========================================================
from agents import Agent, Runner            # the two core primitives

# An Agent = model + a name + instructions (its system prompt in plain English)
assistant = Agent(
    name="Assistant",                                        # used in logs and handoffs
    instructions="You are a concise, friendly helper. Answer in one or two sentences.",
    model=DEFAULT_MINI_MODEL,                                # fast + cheap model for our agents
)

# Runner.run_sync drives the whole agentic loop and returns a result object
result = Runner.run_sync(assistant, "In one sentence, what is an AI agent?")

# result.final_output is the finished answer text
pretty_print(result.final_output, title="Your first agent's answer")

## 🔧 3. Tools with `@function_tool`

An agent that can only talk is a chatbot. An agent that can **act** needs tools. In M3 you hand-wrote a JSON schema for every function. The SDK removes that chore: you decorate an ordinary Python function with **`@function_tool`**, and the SDK reads the function's **name**, its **type-hinted arguments**, and its **docstring** to build the schema for you automatically.

Two rules make this work well:
1. **Add type hints** (`city: str`) — they become the argument schema the model fills in.
2. **Write a real docstring** — it becomes the tool's description, which is how the model decides *when* to call it. Treat it like documentation aimed at the model.

Below we give the agent a `get_weather` tool. When you ask about weather, the Runner will call the tool, get the value, and let the agent finish the answer with real data — the exact call-execute-return loop from M3, now handled for you.

In [ ]:
# ==========================================================
# 3. A tool: a plain function + the @function_tool decorator
# ==========================================================
from agents import function_tool            # the decorator that turns a function into a tool

@function_tool                              # <- this line registers the function as a tool
def get_weather(city: str) -> str:          # type hints become the argument schema
    """Get the current weather for a given city."""   # docstring becomes the tool description
    # In a real app this would call a weather API. We fake it so the lab always runs.
    fake = {                                # a tiny lookup so results are deterministic
        "Boston": "58°F and cloudy",
        "Miami": "84°F and sunny",
        "Seattle": "51°F and rainy",
    }
    return fake.get(city, f"70°F and clear in {city}")     # default for any other city

# Build an agent that is ALLOWED to use the tool (pass it in the tools list)
weather_agent = Agent(
    name="WeatherAssistant",
    instructions="You help with weather questions. Use the get_weather tool when asked about a city.",
    model=DEFAULT_MINI_MODEL,
    tools=[get_weather],                    # the agent may call this tool; the Runner handles the loop
)

# Ask a question that needs the tool. The Runner calls get_weather for us behind the scenes.
result = Runner.run_sync(weather_agent, "What's the weather in Boston?")
pretty_print(result.final_output, title="Tool-grounded answer")

> **Pause and think.** Notice what you did *not* write: no JSON schema, no `tool_calls` parsing, no `role: tool` message, no loop. The SDK read your function's signature and docstring and ran the whole call-execute-return cycle for you. Compare that to the M3 function-calling lab — same mechanics, a fraction of the code. Which tool in your own work would you register first, and what would its one-line docstring say so the model always picks it at the right moment?

**Your notes** *(double-click to edit)*

- A tool from my own work I would register first: 
- Its type-hinted arguments: 
- The one-line docstring that tells the model when to call it: 

## 🔀 4. Handoffs: a triage agent routing to specialists

One giant do-everything agent gets brittle fast. The SDK's signature idea is the **handoff**: build small **specialists**, then a **triage** agent that reads each request and *delegates* to the right one. Routing is decided at runtime, not hard-wired, so the same system handles many kinds of requests without you enumerating every path.

The mechanics are dead simple: an agent lists other agents in its `handoffs=[...]`, and its instructions tell it *when* to route to each. Below, a **Triage** agent routes billing questions to a **Billing** specialist and technical questions to a **Support** specialist. The whole thing is still **one run** — control moves between agents, but `Runner.run_sync` drives it start to finish.

In [ ]:
# ==========================================================
# 4. Handoffs: a triage agent routes to specialist agents
# ==========================================================
# Two small specialists, each with a narrow job described in its instructions
billing_agent = Agent(
    name="Billing",
    instructions="You handle billing questions: invoices, refunds, charges. Be precise and reassuring.",
    model=DEFAULT_MINI_MODEL,
)
support_agent = Agent(
    name="Support",
    instructions="You handle technical support: bugs, errors, how-to. Give clear step-by-step help.",
    model=DEFAULT_MINI_MODEL,
)

# The triage agent does NOT answer itself — it routes to the right specialist
triage_agent = Agent(
    name="Triage",
    instructions="Route the user to the right specialist. Billing questions -> Billing. "
                 "Technical or how-to questions -> Support.",
    model=DEFAULT_MINI_MODEL,
    handoffs=[billing_agent, support_agent],   # the agents Triage is allowed to delegate to
)

# One run, two different requests: watch which specialist ends up answering.
billing_result = Runner.run_sync(triage_agent, "I was charged twice for my subscription this month.")
support_result = Runner.run_sync(triage_agent, "The app crashes every time I click export. How do I fix it?")

pp({
    "billing question -> handled by": billing_result.last_agent.name,   # which agent produced the answer
    "billing answer":                 billing_result.final_output,
    "support question -> handled by": support_result.last_agent.name,
    "support answer":                 support_result.final_output,
}, title="Triage routed each request to the right specialist")

## 🛡️ 5. Guardrails (brief)

Real agents need boundaries. **Guardrails** are checks that run *alongside* an agent and **fail fast**:

- **Input guardrails** validate the request *before* the agent spends tokens on it — e.g. reject off-topic or unsafe input. They apply to the **first** agent that receives the request.
- **Output guardrails** validate the final answer *before* it reaches the user — e.g. block a response that leaks private data. They apply to the agent that produces the **final** output.

You attach them via `input_guardrails=[...]` and `output_guardrails=[...]` on an `Agent`. Each guardrail is a small function (decorated with `@input_guardrail` / `@output_guardrail`) that returns a `GuardrailFunctionOutput` saying whether the tripwire fired. When it fires, the SDK raises an exception and stops the run — so bad input never reaches the model and bad output never reaches the user. The cell below sketches the shape so you recognize it; it does not need to run to make the point.

In [ ]:
# ==========================================================
# 5. Guardrails: the shape of an input check (illustrative)
# ==========================================================
from agents import input_guardrail, GuardrailFunctionOutput

@input_guardrail                                  # runs BEFORE the agent, on the incoming request
def stay_on_topic(ctx, agent, user_input):
    """Trip the guardrail if the user asks about something off-topic."""
    banned = "medical advice"                     # a toy rule; real ones can call a model to judge
    tripped = banned in user_input.lower()        # True -> the guardrail fires and stops the run
    return GuardrailFunctionOutput(
        output_info={"reason": banned} if tripped else {},
        tripwire_triggered=tripped,               # when True, the SDK halts before the model runs
    )

# Attach the guardrail to an agent via input_guardrails (output_guardrails works the same way)
guarded_agent = Agent(
    name="GuardedAssistant",
    instructions="You are a helpful support assistant. Stay on supported topics.",
    model=DEFAULT_MINI_MODEL,
    input_guardrails=[stay_on_topic],             # checked on every request before the agent answers
)

# A safe request passes the guardrail and runs normally.
safe = Runner.run_sync(guarded_agent, "How do I reset my password?")
pretty_print(safe.final_output, title="Guardrail passed — request was on-topic")

## 🛠️ 6. Hands-on: build your own tool-using agent

Put it together. Write **one** new tool with `@function_tool`, attach it to an agent, and ask a question that should trigger it. Ideas: `convert_currency(amount, from_ccy, to_ccy)`, `word_count(text)`, or `days_until(date)`. Remember the two rules: **type hints** for the arguments, and a **clear docstring** so the model knows when to call it. Replace each `-----` with the right value.

In [ ]:
# ==========================================================
# 6. Hands-on: your own @function_tool + agent (fill in the -----)
# ==========================================================
@function_tool
def my_tool(-----: -----) -> -----:         # name your argument(s) and add type hints
    """-----"""                              # a clear docstring so the model knows when to call it
    return -----                            # return a value the agent can use in its answer

# Build an agent that is allowed to use your tool
my_agent = Agent(
    name="-----",                           # any name you like
    instructions="-----",                   # tell it its job and when to use your tool
    model=DEFAULT_MINI_MODEL,
    tools=[-----],                          # pass your tool function here
)

# Ask a question that should make the agent call your tool
result = Runner.run_sync(my_agent, "-----")
pretty_print(result.final_output, title="Your agent in action")

## 🎯 Wrap-up

You built agents with the OpenAI Agents SDK using its whole vocabulary:

- **Agent** — a model plus a name and instructions, defined in a couple of lines.
- **Runner** — `Runner.run_sync(agent, "...")` drives the agentic loop and returns `result.final_output`.
- **Tools** — a plain Python function plus `@function_tool`; the SDK builds the schema from your type hints and docstring, no hand-written JSON.
- **Handoffs** — a triage agent lists specialists in `handoffs=[...]` and routes each request at runtime, all inside one run.
- **Guardrails** — `input_guardrails` / `output_guardrails` fail fast on unsafe or off-topic input and output.

Compare this to the raw function-calling loop you wrote by hand in Module 3: same mechanics, a fraction of the code. That is the trade the SDK makes — you describe *what* your system is, and the Runner handles *how* it executes. Next module you tie this together with MCP (agent-to-tools) and A2A (agent-to-agent) into a full production system.